# Сравнение на основе сохраненных метрик (JSON)


In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "README.md").exists()
    and (path / "Data_making").is_dir()
    and (path / "Models").is_dir()
)

In [ ]:
import json

base_path = PROJECT_ROOT / "Models" / "Compare models"
logreg_path = base_path / 'logreg_metrics.json'
cat_path = base_path / 'catboost_metrics.json'

logreg_metrics = json.loads(logreg_path.read_text())
cat_metrics = json.loads(cat_path.read_text())

metrics_df = pd.DataFrame([logreg_metrics, cat_metrics])
metrics_df['Feature Scenario'] = metrics_df['feature_scenario']
metrics_df['N Features'] = metrics_df['feature_columns'].apply(len)
metrics_df

In [ ]:
# Визуализация сохраненных метрик (устойчиво к разным ключам JSON)
# Берем только общие и числовые метрики для честного сравнения.
metric_order = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']

available_metrics = [
    m for m in metric_order
    if m in metrics_df.columns and pd.api.types.is_numeric_dtype(metrics_df[m])
]

if not available_metrics:
    raise ValueError('Не найдены общие числовые метрики для сравнения.')

metrics_plot_df = metrics_df[['model'] + available_metrics].copy()
metrics_long = metrics_plot_df.melt(id_vars='model', var_name='metric', value_name='value')

fig = px.bar(
    metrics_long,
    x='metric', y='value', color='model', barmode='group', text='value',
    category_orders={'metric': available_metrics},
    title='Сравнение ключевых метрик моделей'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis=dict(range=[0, 1]))
fig.show()

# Доп. таблица: поля, которые не участвуют в прямом сравнении
extra_cols = [c for c in metrics_df.columns if c not in ['model'] + available_metrics]
if extra_cols:
    print('Служебные/дополнительные поля (не для bar-сравнения):')
    display(metrics_df[['model'] + extra_cols])


In [ ]:
# Основная сравнительная таблица без искусственного агрегата mean_score.
# Метрики интерпретируются отдельно; mean_score не является финальным критерием.
comparison_table = metrics_df[
    ['model', 'Feature Scenario', 'N Features'] + available_metrics
].copy()
comparison_table


## Вывод

Модели сравниваются на едином сценарии `common_no_avg_grade` и одинаковых 12 признаках. AUC, Accuracy, Precision, Recall и F1 рассматриваются отдельно; искусственный агрегат `mean_score` не используется как финальный критерий.